# 01 · Análisis exploratorio de PM2.5 en el Valle de Aburrá

**Fuente de los datos.** Conjunto [Calidad del Aire en Colombia](https://www.datos.gov.co/Ambiente-y-Desarrollo-Sostenible/Calidad-del-Aire-en-Colombia/g4t8-zkc3) (SISAIRE, IDEAM) de datos.gov.co, filtrado por la autoridad ambiental AMVA y la variable PM2.5. Son mediciones horarias en µg/m³.

**Antes de ejecutar:** `python -m src.download_data` y `python -m src.preprocess`.

El preprocesamiento (`src/preprocess.py`) descarta valores imposibles, elimina duplicados y calcula el promedio diario solo para los días con al menos 18 de 24 horas medidas. Este notebook parte de esa tabla diaria.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config, features, ica, preprocess

plt.rcParams.update({"figure.figsize": (10, 3.8), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

diario = preprocess.cargar_diario()
estaciones = preprocess.cargar_estaciones()
print(f"{len(diario):,} días-estación · {diario['estacion'].nunique()} estaciones · {diario['fecha'].min():%Y-%m-%d} a {diario['fecha'].max():%Y-%m-%d}")
diario.head()

## 1. Cobertura: ¿cuántos días válidos tiene cada estación?

Una estación con muchos vacíos aporta menos ejemplos al modelo y más días sin predicción en el dashboard.

In [ ]:
total_dias = (diario["fecha"].max() - diario["fecha"].min()).days + 1
cobertura = (diario.groupby("estacion").size() / total_dias).sort_values()
ax = cobertura.plot.barh(figsize=(10, 0.32 * len(cobertura) + 1), color="#1F4E79")
ax.set_xlabel("Fracción de días con promedio válido"); ax.set_ylabel(""); ax.set_xlim(0, 1)
plt.show()
estaciones[["estacion", "municipio", "dias_validos", "primer_dia", "ultimo_dia"]].sort_values("dias_validos", ascending=False)

## 2. Distribución del PM2.5 diario y de las categorías ICA

Las líneas verticales son los puntos de corte del ICA para PM2.5 de 24 horas (Resolución 2254 de 2017): 12, 37 y 55 µg/m³.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.8))
a.hist(diario["pm25"], bins=60, color="#1F4E79")
for corte in features.BORDES_CLASE:
    a.axvline(corte, color="#555", ls="--", lw=1)
a.set_xlabel("PM2.5 diario (µg/m³)"); a.set_ylabel("Días-estación")

conteo = diario["categoria"].value_counts().reindex(ica.NOMBRES, fill_value=0)
b.bar(range(len(conteo)), conteo.values, color=[ica.COLORES[c] for c in conteo.index])
b.set_xticks(range(len(conteo))); b.set_xticklabels([c.replace(" para ", "\npara ") for c in conteo.index], fontsize=8)
b.set_ylabel("Días-estación")
plt.tight_layout(); plt.show()
(conteo / conteo.sum()).round(4).to_frame("fracción")

## 3. Estacionalidad

En el valle se habla de dos temporadas de episodios, asociadas a la transición entre época seca y de lluvias. Aquí se comprueba con los datos qué meses concentran los valores altos; de esto sale la variable `temporada` del modelo.

In [ ]:
valle = diario.groupby("fecha")["pm25"].mean()
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [2, 1]})
valle.rolling(7, min_periods=3).mean().plot(ax=a, color="#1F4E79")
for corte in features.BORDES_CLASE:
    a.axhline(corte, color="#555", ls="--", lw=0.8)
a.set_ylabel("Promedio del valle, media móvil 7 días (µg/m³)"); a.set_xlabel("")

por_mes = [diario.loc[diario["fecha"].dt.month == m, "pm25"].to_numpy() for m in range(1, 13)]
b.boxplot(por_mes, showfliers=False)
b.set_xlabel("Mes"); b.set_ylabel("PM2.5 diario (µg/m³)")
plt.tight_layout(); plt.show()

episodio = diario.assign(episodio=features.clase_de(diario["pm25"]) >= ica.CLASE_EPISODIO)
episodio.groupby(episodio["fecha"].dt.month)["episodio"].mean().round(3).to_frame("fracción de días en episodio").T

## 4. Día de la semana

Si el tráfico influye, los fines de semana deberían ser más limpios. De aquí sale la variable `dia_manana`.

In [ ]:
dias = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
por_dia = diario.groupby(diario["fecha"].dt.day_of_week)["pm25"].mean()
ax = por_dia.plot.bar(color=["#1F4E79"] * 5 + ["#9AA0A6"] * 2)
ax.set_xticklabels(dias, rotation=0); ax.set_xlabel(""); ax.set_ylabel("PM2.5 promedio (µg/m³)")
plt.show()

## 5. Persistencia: ¿cuánto se parece mañana a hoy?

Esta es la pregunta clave para el modelo. Si la clase de mañana casi siempre es la de hoy, la línea base de persistencia será difícil de superar y el valor del modelo estará en anticipar los cambios de categoría.

In [ ]:
tabla = features.construir_tabla(diario).dropna(subset=["y"])
print(f"Correlación entre PM2.5 de hoy y de mañana: {tabla['pm25_hoy'].corr(tabla['pm25_manana']):.3f}")
print(f"Días en que mañana tiene la misma clase que hoy: {(tabla['clase_hoy'] == tabla['y']).mean():.1%}")

transicion = pd.crosstab(tabla["clase_hoy"], tabla["y"].astype(int), normalize="index")
transicion.index = [ica.CLASES[i] for i in transicion.index]
transicion.columns = [ica.CLASES[i] for i in transicion.columns]
print("\nP(clase de mañana | clase de hoy):")
transicion.round(3)

In [ ]:
muestra = tabla.sample(min(len(tabla), 6000), random_state=0)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(muestra["pm25_hoy"], muestra["pm25_manana"], s=4, alpha=0.25, color="#1F4E79")
limite = float(max(muestra["pm25_hoy"].max(), muestra["pm25_manana"].max()))
ax.plot([0, limite], [0, limite], color="#555", lw=1, ls="--")
ax.set_xlabel("PM2.5 hoy (µg/m³)"); ax.set_ylabel("PM2.5 mañana (µg/m³)")
plt.show()

## 6. Qué se lleva el modelo de este análisis

- **Objetivo:** la clase ICA del día siguiente. Las categorías superiores a «Dañina para grupos sensibles» son muy escasas en promedios diarios, por eso se agrupan en «Dañina o peor».
- **Variables:** nivel de hoy, clase de ayer y tendencia (persistencia, sección 5); temporada (sección 3); tipo de día (sección 4); nivel promedio del valle (contexto regional).
- **Línea base exigente:** la persistencia (sección 5).
- **Evaluación:** partición temporal, porque los días consecutivos están muy correlacionados y una partición aleatoria filtraría información.

El modelo, sus métricas y sus limitaciones están en `02_modelo.ipynb`.